In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os, json, cv2
import numpy as np
import pandas as pd
import rasterio
from tqdm import tqdm


In [3]:
BASE_RAW_DIR = "/content/drive/MyDrive/disaster_data/raw"
OUT_DIR = "/content/drive/MyDrive/disaster_data/dataset"
IMG_DIR = f"{OUT_DIR}/images"

os.makedirs(IMG_DIR, exist_ok=True)


In [4]:
def load_annotations(json_path):
    with open(json_path) as f:
        data = json.load(f)

    if isinstance(data, dict):
        if "features" in data:
            return data["features"]
        if "annotations" in data:
            return data["annotations"]
        raise ValueError("Unknown JSON format")

    return data


In [5]:
def extract_building(image, pixels,
                     output_size=128,
                     pad_ratio=0.25,
                     min_ratio=0.15):

    pts = np.array([(p["x"], p["y"]) for p in pixels], dtype=np.int32)
    x1, y1 = pts.min(axis=0)
    x2, y2 = pts.max(axis=0)

    w, h = x2-x1, y2-y1
    if w <= 0 or h <= 0:
        return None

    px, py = int(w*pad_ratio), int(h*pad_ratio)
    x1 = max(0, x1-px); y1 = max(0, y1-py)
    x2 = min(image.shape[1], x2+px)
    y2 = min(image.shape[0], y2+py)

    crop = image[y1:y2, x1:x2].copy()

    mask = np.zeros(crop.shape[:2], np.uint8)
    cv2.fillPoly(mask, [pts - [x1,y1]], 255)

    if cv2.countNonZero(mask) / mask.size < min_ratio:
        return None

    mean_color = crop[mask > 0].mean(axis=0)
    crop[mask == 0] = mean_color

    return cv2.resize(crop, (output_size, output_size))


In [6]:
records = []
img_id = 0

for scene in sorted(os.listdir(BASE_RAW_DIR)):
    scene_path = os.path.join(BASE_RAW_DIR, scene)
    if not os.path.isdir(scene_path):
        continue

    print("Processing:", scene)

    # find files
    tiff = next((f for f in os.listdir(scene_path) if f.endswith(".tif")), None)
    jsn  = next((f for f in os.listdir(scene_path) if f.endswith(".json")), None)

    if tiff is None or jsn is None:
        print("  ❌ Missing TIFF/JSON")
        continue

    with rasterio.open(os.path.join(scene_path, tiff)) as src:
        image = src.read([1,2,3]).transpose(1,2,0)

    anns = load_annotations(os.path.join(scene_path, jsn))

    for ann in tqdm(anns, leave=False):
        if not isinstance(ann, dict):
            continue

        if ann.get("label") not in ["no damage", "minor damage", "destroyed"]:
            continue

        crop = extract_building(image, ann["pixels"])
        if crop is None:
            continue

        label = 1 if ann["label"] == "destroyed" else 0

        name = f"{img_id:07d}.png"
        cv2.imwrite(os.path.join(IMG_DIR, name),
                    cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))

        records.append({"image": name, "label": label})
        img_id += 1

print("✅ Total extracted images:", img_id)


Processing: scene_01


Processing: scene_02


Processing: scene_03


Processing: scene_04


Processing: scene_05


✅ Total extracted images: 100


In [7]:
OUT_DIR = "/content/drive/MyDrive/disaster_data/dataset"
IMG_DIR = f"{OUT_DIR}/images"

import os
os.makedirs(IMG_DIR, exist_ok=True)


In [8]:
!ls $IMG_DIR | head


0000000.png
0000001.png
0000002.png
0000003.png
0000004.png
0000005.png
0000006.png
0000007.png
0000008.png
0000009.png


In [9]:
df = pd.DataFrame(records)
df.to_csv(f"{OUT_DIR}/labels_binary.csv", index=False)

from sklearn.model_selection import train_test_split
train, val = train_test_split(
    df, test_size=0.2,
    stratify=df.label, random_state=42
)

train.to_csv(f"{OUT_DIR}/train_binary.csv", index=False)
val.to_csv(f"{OUT_DIR}/val_binary.csv", index=False)

print(df.label.value_counts())


label
0    61
1    39
Name: count, dtype: int64


In [10]:
import torch
from torch.utils.data import Dataset

class BuildingDataset(Dataset):
    def __init__(self, csv, img_dir):
        self.df = pd.read_csv(csv)
        self.img_dir = img_dir

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = cv2.imread(os.path.join(self.img_dir, row.image))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) / 255.0
        img = torch.tensor(img, dtype=torch.float32).permute(2,0,1)
        return img, torch.tensor(row.label)


In [11]:
import torch.nn as nn
import torch.nn.functional as F

class BestBinaryCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128,256,3,padding=1), nn.BatchNorm2d(256), nn.ReLU(),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256,2)

    def forward(self,x):
        x = self.features(x)
        x = self.gap(x).flatten(1)
        return self.fc(x)


In [12]:
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"

train_ds = BuildingDataset(f"{OUT_DIR}/train_binary.csv", IMG_DIR)
val_ds   = BuildingDataset(f"{OUT_DIR}/val_binary.csv", IMG_DIR)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=16)

model = BestBinaryCNN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()


In [13]:
for epoch in range(30):
    model.train()
    total = 0
    for x,y in train_loader:
        x,y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()
        total += loss.item()
    print(f"Epoch {epoch+1:02d} | Loss {total:.3f}")


Epoch 01 | Loss 2.027
Epoch 02 | Loss 0.813
Epoch 03 | Loss 0.540
Epoch 04 | Loss 0.606
Epoch 05 | Loss 0.475
Epoch 06 | Loss 0.743
Epoch 07 | Loss 0.571
Epoch 08 | Loss 0.940
Epoch 09 | Loss 0.362
Epoch 10 | Loss 0.351
Epoch 11 | Loss 0.325
Epoch 12 | Loss 0.532
Epoch 13 | Loss 0.511
Epoch 14 | Loss 0.276
Epoch 15 | Loss 0.644
Epoch 16 | Loss 0.413
Epoch 17 | Loss 0.221
Epoch 18 | Loss 0.174
Epoch 19 | Loss 0.195
Epoch 20 | Loss 0.269
Epoch 21 | Loss 0.536
Epoch 22 | Loss 1.002
Epoch 23 | Loss 0.872
Epoch 24 | Loss 0.344
Epoch 25 | Loss 0.231
Epoch 26 | Loss 0.254
Epoch 27 | Loss 0.418
Epoch 28 | Loss 0.435
Epoch 29 | Loss 0.293
Epoch 30 | Loss 0.427


In [14]:
import torch
from torch.utils.data import Dataset
import cv2

class BuildingDataset(Dataset):
    def __init__(self, csv_file, img_dir):
        self.data = pd.read_csv(csv_file)
        self.img_dir = img_dir

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img = cv2.imread(os.path.join(self.img_dir, row.image))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img / 255.0
        img = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1)
        label = torch.tensor(row.label, dtype=torch.long)
        return img, label


In [15]:
import torch.nn as nn
import torch.nn.functional as F

class DamageCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool = nn.MaxPool2d(2,2)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = self.gap(x).squeeze(-1).squeeze(-1)
        return self.fc(x)


In [16]:
import numpy as np
import pandas as pd
import torch
from sklearn.utils.class_weight import compute_class_weight

train_df = pd.read_csv("/content/drive/MyDrive/disaster_data/dataset/train_binary.csv")

labels = train_df["label"].values

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float32)
print("Class weights:", class_weights)


Class weights: tensor([0.8163, 1.2903])


In [17]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = DamageCNN().to(device)

criterion = nn.CrossEntropyLoss()   # ✅ no weights
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [18]:
df["label"].value_counts()


,count
label,
0,61
1,39


In [19]:
for epoch in range(20):
    model.train()
    total_loss = 0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1:02d} | Train Loss: {total_loss:.4f}")


Epoch 01 | Train Loss: 4.8949
Epoch 02 | Train Loss: 3.6096
Epoch 03 | Train Loss: 3.3960
Epoch 04 | Train Loss: 3.3792
Epoch 05 | Train Loss: 3.4306
Epoch 06 | Train Loss: 3.4520
Epoch 07 | Train Loss: 3.3063
Epoch 08 | Train Loss: 3.4213
Epoch 09 | Train Loss: 3.4381
Epoch 10 | Train Loss: 3.6334
Epoch 11 | Train Loss: 3.4080
Epoch 12 | Train Loss: 3.3333
Epoch 13 | Train Loss: 3.3218
Epoch 14 | Train Loss: 3.2464
Epoch 15 | Train Loss: 3.1506
Epoch 16 | Train Loss: 3.1373
Epoch 17 | Train Loss: 3.1871
Epoch 18 | Train Loss: 3.0731
Epoch 19 | Train Loss: 2.8735
Epoch 20 | Train Loss: 2.8925


In [20]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        preds = model(imgs).argmax(dim=1).cpu().numpy()
        y_true.extend(labels.numpy())
        y_pred.extend(preds)

print(classification_report(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))


              precision    recall  f1-score   support

           0       0.91      0.83      0.87        12
           1       0.78      0.88      0.82         8

    accuracy                           0.85        20
   macro avg       0.84      0.85      0.85        20
weighted avg       0.86      0.85      0.85        20

[[10  2]
 [ 1  7]]
